In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt

# 15 - Merge RegBL and Geology

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `04_SWISSTOPO_geology`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

### Geology

In [ ]:
# Read useful layers.
GEOL_base = gpd.read_file(f'{ROOT}data/raw/04_SWISSTOPO_geology/PY_Surface_Base.shp').to_crs('EPSG:2056')
GEOL_tecto = gpd.read_file(f'{ROOT}data/raw/04_SWISSTOPO_geology/LI_Accident_tecto.shp').to_crs('EPSG:2056')

### Current dataset

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.11.csv', index_col=0)

df_regbl.head()

## Convert main DataFrame to GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

## Select only the needed columns

In [ ]:
GEOL_base_filt = GEOL_base[[
    'LITHO',
    'PERIODE',
    'EPOQUE',
    'ETAGE',
    'LEG_TEC_1',
    'LEG_TEC_2',
    'LEG_TEC_3',
    'AQUIFERE',
    'HYDROGEOL',
    'ID_LITH',
    'LITH_PET',
    'ID_GENESE',
    'GENESE',
    'ID_TYP_ROC',
    'TYPE_ROCHE',
    'COUVERTURE',
    'geometry'
]];

GEOL_tecto_filt = GEOL_tecto[['Type', 'ID', 'Shape_Leng', 'HYDRO','geometry']];

## Perform spatial joint

In [ ]:
if "index_right" in df_regbl.columns:
    df_regbl = df_regbl.drop(columns="index_right")

In [ ]:
gdf_geol = df_regbl.sjoin(GEOL_base_filt, how='left')

## Selection of the faults detected "certainly" by Swisstopo and geologists

In [ ]:
GEOL_faults = GEOL_tecto_filt[GEOL_tecto_filt['Type'].str.contains('certain', case=False)]
type(GEOL_faults)

## Calculate distance to fault

In [ ]:
# Convert to a multiline string using union_all()

GEOL_tecto_geos = GEOL_faults.union_all()

In [ ]:
# Calculate distance to line (therefore shortest one since only one line exist)

dist = gdf_geol.geometry.distance(GEOL_tecto_geos)

gdf_geol = gdf_geol.assign(Distance_to_fault=dist)

## Rename columns

In [ ]:
cols = ['LITHO',
    'PERIODE',
    'EPOQUE',
    'ETAGE',
    'LEG_TEC_1',
    'LEG_TEC_2',
    'LEG_TEC_3',
    'AQUIFERE',
    'HYDROGEOL',
    'ID_LITH',
    'LITH_PET',
    'ID_GENESE',
    'GENESE',
    'ID_TYP_ROC',
    'TYPE_ROCHE',
    'COUVERTURE',
    'Type', 
    'ID', 
    'Shape_Leng', 
    'HYDRO'
    ]

# Add 'Geol - ' to the beginning of each column name
for col in cols:
    gdf_geol.rename(columns={col: 'Geol - ' + col}, inplace=True)
    

gdf_geol = gdf_geol.rename(columns={'Distance_to_fault': 'Geol - Distance to fault (m)'})


In [ ]:
gdf_geol.head()

## Convert back to DataFrame

In [ ]:
## Convert back to a dataframe
df_regbl = gdf_geol.drop(columns='geometry')

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.12.csv")